In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
DATASET_PATH = 'data'
IMG_HEIGHT = 40
IMG_WIDTH = 150
LABEL_LENGTH = 5

images = []
labels = []
    
for filename in os.listdir(DATASET_PATH):
    if filename.endswith('.jpg'):
        img_path = os.path.join(DATASET_PATH, filename)
        img = Image.open(img_path)
        images.append(np.array(img))
            
        label = filename[:-4]
        labels.append(label)

num_images = len(images)
print(f"Total number of images: {num_images}")

label_lengths = [len(label) for label in labels]
print(f"All labels are of length 5: {all(length == LABEL_LENGTH for length in label_lengths)}")
image_shapes = [img.shape for img in images]
image_heights = [shape[0] for shape in image_shapes]
image_widths = [shape[1] for shape in image_shapes]
print(f"All images are of size {IMG_HEIGHT}x{IMG_WIDTH}: {all(shape[0] == IMG_HEIGHT and shape[1] == IMG_WIDTH for shape in image_shapes)}")

# Randomly display 5 images with their labels and return the indices
indices = np.random.choice(len(images), 5, replace=False)
plt.figure(figsize=(15, 5))
for i, idx in enumerate(indices):
    plt.subplot(1, 5, i+1)
    plt.imshow(images[idx])
    plt.title(labels[idx])
    plt.axis('off')
plt.show()


NameError: name 'os' is not defined

In [ ]:
# Convert to grayscale and normalize
images = np.array(images)

if images.ndim == 4 and images.shape[3] == 3:
    processed = np.dot(images[...,:3], [0.2989, 0.5870, 0.1140])
else:
    processed = images

processed = np.array(processed) / 255.0
if processed.ndim == 3:
    processed = np.expand_dims(processed, axis=-1)
print(f"Processed images shape: {processed.shape}")
# Double check by displaying the same indices
plt.figure(figsize=(15, 5))
for i, idx in enumerate(indices):
    plt.subplot(1, 5, i+1)
    plt.imshow(processed[idx].squeeze(), cmap='gray')
    plt.title(labels[idx])
    plt.axis('off')
plt.show()

In [ ]:
# Encode labels using one-hot encoding and visualize character frequency
char_set = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
le = LabelEncoder()
le.fit(list(char_set))

encoded_labels = np.zeros((len(labels), LABEL_LENGTH, len(char_set)))

for i, label in enumerate(labels):
    chars = list(label.upper())

    for j, char in enumerate(chars):
        char_idx = le.transform([char])[0]
        encoded_labels[i, j, char_idx] = 1

char_counts = encoded_labels.sum(axis=(0, 1))

plt.figure(figsize=(12, 5))
sns.barplot(x=list(char_set), y=char_counts)
plt.xlabel("Character")
plt.ylabel("Frequency")
plt.title("Character frequency across dataset")
plt.show()


In [ ]:
# Define the CNN model with multiple output heads
VOCAB_SIZE = 36 # Assuming 26 lowercase letters + 10 digits
inputs = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))

x = Conv2D(32, (3,3), activation='relu')(inputs)
x = MaxPooling2D((2,2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)

output_heads = [Dense(VOCAB_SIZE, activation='softmax', name=f'char_{i+1}')(x) for i in range(LABEL_LENGTH)]
model = Model(inputs=inputs, outputs=output_heads)

model.summary()